In [3]:
import sys
sys.path.append("..")
from unsloth import FastLanguageModel
from src.utils.config_loader import load_config
import pandas as pd
import torch
from tqdm import tqdm

cfg = load_config("../configs/config.yaml")

In [4]:
# gpt-oss-20b-bnb-4bit 大约占用 14GB 显存，5090 (32GB) 完全够用
# 从已加载的配置中读取教师模型 ID
teacher_model_id = cfg.model_teacher.model_id
load_in_4bit = cfg.model_teacher.load_in_4bit


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=teacher_model_id,
    max_seq_length=2048,
    load_in_4bit=load_in_4bit,
    dtype=None
)
FastLanguageModel.for_inference(model)

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /unsloth/gpt-oss-20b-GGUF/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x71f538e53790>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 00ceeffe-62c1-45f6-8e14-bafa47864a7c)')' thrown while requesting HEAD https://huggingface.co/unsloth/gpt-oss-20b-GGUF/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /unsloth/gpt-oss-20b-GGUF/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x71f538e53790>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 00ceeffe-62c1-45f6-8e14-bafa47864a7c)')' thrown while requesting HEAD https://huggingface.co/unsloth/gpt-oss-20b-GGUF/resolve/main/config.json
Retry

KeyboardInterrupt: 

In [ ]:
# Cell 3: 定义蒸馏 Prompt
# 我们给 Teacher 真实标签，让它生成“原因” (Rationale)
DISTILL_PROMPT = """You are an expert linguist.
Sentence: "{text}"
Aspect: "{aspect}"
Sentiment: "{polarity}"

Please explain concisely step-by-step why the sentiment towards '{aspect}' is '{polarity}' based on the sentence structure and adjectives used.
Format your output as: <think>... explanation... </think>"""

In [ ]:
df_rest_train = pd.read_json("../data/processed/train_rest_clean.jsonl", lines=True)
df_lap_train = pd.read_json("../data/processed/train_lapt_clean.jsonl", lines=True)

In [ ]:
augmented_rest_data = []
augmented_lap_data = []
max_new_tokens = cfg.model_teacher.max_new_tokens
temperature = cfg.model_teacher.temperature

print("Generating Rest Rationales...")
for _, row in tqdm(df_rest_train.iterrows(), total=len(df_rest_train)):
    user_prompt = DISTILL_PROMPT.format(
        text=row['text'], 
        aspect=row, 
        polarity=row['polarity']
    )
    
    messages = [{"role": "user", "content": user_prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    
    # gpt-oss 推理配置
    outputs = model.generate(
        inputs, 
        max_new_tokens=max_new_tokens, 
        temperature=temperature
    )
    output_text = tokenizer.decode(outputs, skip_special_tokens=True)
    
    # 提取生成的解释部分 (假设 Teacher 遵循指令)
    # 这里的解析逻辑取决于 gpt-oss 的具体输出，可能需要简单 split
    assistant_reply = output_text.split("assistant")[-1].strip()
    
    # 构建新的训练样本：
    # Input: Sentence + Aspect
    # Output: Teacher生成的解释 + 真实标签
    new_sample = {
        "text": row['text'],
        "aspect": row,
        # 这是给 Student 学习的目标：先思考，再输出标签
        "target_output": f"{assistant_reply}\nFinal Sentiment: {row['polarity']}"
    }
    augmented_rest_data.append(new_sample)
    
print("Generating Laptop Rationales...")
for _, row in tqdm(df_lap_train.iterrows(), total=len(df_lap_train)):
    user_prompt = DISTILL_PROMPT.format(
        text=row['text'], 
        aspect=row, 
        polarity=row['polarity']
    )
    
    messages = [{"role": "user", "content": user_prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    
    # gpt-oss 推理配置
    outputs = model.generate(
        inputs, 
        max_new_tokens=max_new_tokens, 
        temperature=temperature
    )
    output_text = tokenizer.decode(outputs, skip_special_tokens=True)
    
    # 提取生成的解释部分 (假设 Teacher 遵循指令)
    # 这里的解析逻辑取决于 gpt-oss 的具体输出，可能需要简单 split
    assistant_reply = output_text.split("assistant")[-1].strip()
    
    # 构建新的训练样本：
    # Input: Sentence + Aspect
    # Output: Teacher生成的解释 + 真实标签
    new_sample = {
        "text": row['text'],
        "aspect": row,
        # 这是给 Student 学习的目标：先思考，再输出标签
        "target_output": f"{assistant_reply}\nFinal Sentiment: {row['polarity']}"
    }
    augmented_lap_data.append(new_sample)

In [ ]:
pd.DataFrame(augmented_rest_data).to_json("../data/processed/train_rest_cot_distilled.jsonl", orient='records', lines=True)
pd.DataFrame(augmented_lap_data).to_json("../data/processed/train_lap_cot_distilled.jsonl", orient='records', lines=True)
# 释放显存
del model, tokenizer
torch.cuda.empty_cache()